# First-Ensure transition analysis

Align each recording to its first embedded Ensure TTL and display 60 seconds before through 60 seconds after delivery. The slow stage reads each `.doric` file once and saves a reusable cache containing lifetime, licking, and all Ensure events in the window.

In [ ]:
from datetime import date, datetime
from pathlib import Path
from time import perf_counter
import csv
import matplotlib.pyplot as plt
import numpy as np
from openpyxl import load_workbook
from fluopulse_analysis import extract_perievent, find_doric_files, read_doric

## Configuration

The baseline is the mean lifetime from −30 to −5 seconds relative to the first Ensure delivery. Recordings with partial pre-event coverage are retained and reported in `recording_qc.csv`.

In [ ]:
SESSION_TABLE = Path.home() / "Documents" / "SC_FLiPmice_2025.xlsx"
DORIC_ROOT = Path(r"Z:\FLIM FLIP")
OUTPUT_FOLDER = Path.cwd() / "analysis_output_first_ensure"
CACHE_FILE = OUTPUT_FOLDER / "first_ensure_cache.npz"
PERIEVENT_WINDOW_S = (-60.0, 60.0)
BASELINE_WINDOW_S = (-30.0, -5.0)
SAMPLE_INTERVAL_S = 0.1
MIN_BASELINE_COVERAGE = 0.8
print(f"Doric root available: {DORIC_ROOT.is_dir()} ({DORIC_ROOT})")

In [ ]:
def normalize_date(value):
    if isinstance(value, (date, datetime)):
        return value.strftime("%y%m%d")
    return str(int(value)).zfill(6)

workbook = load_workbook(SESSION_TABLE, data_only=True, read_only=True)
worksheet = workbook.active
rows = list(worksheet.iter_rows(values_only=True))
workbook.close()
headers = [str(value).strip() if value is not None else "" for value in rows[0]]
if not headers or headers[0].casefold() != "mouse":
    raise ValueError("The first workbook column must be named 'mouse'")
session_records = []
for row in rows[1:]:
    if not row or row[0] is None:
        continue
    mouse = str(row[0]).strip()
    for column, value in zip(headers[1:], row[1:], strict=False):
        if value is not None:
            session_records.append({"mouse": mouse, "date": normalize_date(value), "source_column": column})
print(f"Loaded {len(session_records)} mouse-date sessions")

## Slow processing and cache creation

Run this cell once, and rerun it only when source files or processing-window settings change.

In [ ]:
def recording_label(path, mouse, session_date):
    stem = path.stem
    for prefix in (f"{mouse}_{session_date}_", f"{session_date}_{mouse}_"):
        if stem.casefold().startswith(prefix.casefold()):
            return stem[len(prefix):]
    return stem

def write_dict_rows(path, rows):
    if not rows:
        return
    fieldnames = []
    for row in rows:
        for key in row:
            if key not in fieldnames:
                fieldnames.append(key)
    with path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

qc_rows = []
metadata_rows = []
tau_rows = []
delta_tau_rows = []
lick_count_rows = []
ensure_count_rows = []
relative_time = None
processing_started = perf_counter()
total_sessions = len(session_records)
print(f"Starting {total_sessions} mouse-date sessions", flush=True)
for session_index, record in enumerate(session_records, start=1):
    mouse = record["mouse"]
    session_date = record["date"]
    print(f"[{session_index}/{total_sessions}] {mouse} {session_date}: searching...", flush=True)
    doric_files = find_doric_files(mouse, session_date, doric_root=DORIC_ROOT)
    if not doric_files:
        qc_rows.append({"mouse": mouse, "date": session_date, "recording": "", "status": "missing .doric"})
        print("    no matching .doric files", flush=True)
        continue
    print(f"    found {len(doric_files)} recording(s)", flush=True)
    for file_index, doric_file in enumerate(doric_files, start=1):
        label = recording_label(doric_file, mouse, session_date)
        file_started = perf_counter()
        print(f"    ({file_index}/{len(doric_files)}) {label}: reading...", flush=True)
        try:
            recording = read_doric(doric_file)
            ensure_pulses = recording.digital_pulses.get("ensure")
            lick_pulses = recording.digital_pulses.get("licking")
            ensure_times = np.array([]) if ensure_pulses is None else ensure_pulses.onset_times
            lick_times = np.array([]) if lick_pulses is None else lick_pulses.onset_times
            if ensure_times.size == 0:
                qc_rows.append({"mouse": mouse, "date": session_date, "recording": label, "status": "no Ensure events"})
                print("        skipped: no Ensure events", flush=True)
                continue
            first_ensure_s = float(ensure_times[0])
            relative_time, tau_trials = extract_perievent(
                recording.time,
                recording.tau_ns,
                np.asarray([first_ensure_s]),
                window_seconds=PERIEVENT_WINDOW_S,
                sample_interval_seconds=SAMPLE_INTERVAL_S,
            )
            relative_time[np.isclose(relative_time, 0.0, atol=SAMPLE_INTERVAL_S / 1000)] = 0.0
            tau_trace = tau_trials[0]
            baseline_mask = (relative_time >= BASELINE_WINDOW_S[0]) & (relative_time < BASELINE_WINDOW_S[1])
            baseline_coverage = float(np.mean(np.isfinite(tau_trace[baseline_mask])))
            if baseline_coverage == 0:
                raise ValueError("No finite lifetime samples in the baseline window")
            baseline_tau = float(np.nanmean(tau_trace[baseline_mask]))
            delta_tau = tau_trace - baseline_tau
            bin_edges = np.concatenate([
                relative_time - SAMPLE_INTERVAL_S / 2,
                [relative_time[-1] + SAMPLE_INTERVAL_S / 2],
            ])
            def aligned_counts(event_times):
                relative_events = np.asarray(event_times, dtype=float) - first_ensure_s
                relative_events = relative_events[
                    (relative_events >= relative_time[0]) & (relative_events <= relative_time[-1])
                ]
                return np.histogram(relative_events, bins=bin_edges)[0]
            lick_counts = aligned_counts(lick_times)
            ensure_counts = aligned_counts(ensure_times)
            status = "ok" if baseline_coverage >= MIN_BASELINE_COVERAGE else "limited baseline coverage"
            qc_rows.append({
                "mouse": mouse,
                "date": session_date,
                "recording": label,
                "status": status,
                "first_ensure_s": first_ensure_s,
                "baseline_coverage": baseline_coverage,
                "lick_count_in_window": int(lick_counts.sum()),
                "ensure_count_in_window": int(ensure_counts.sum()),
            })
            metadata_rows.append({
                "mouse": mouse,
                "date": session_date,
                "recording": label,
                "source_file": str(doric_file),
                "first_ensure_s": first_ensure_s,
                "baseline_tau_ns": baseline_tau,
                "baseline_coverage": baseline_coverage,
            })
            tau_rows.append(tau_trace)
            delta_tau_rows.append(delta_tau)
            lick_count_rows.append(lick_counts)
            ensure_count_rows.append(ensure_counts)
            print(f"        cached first Ensure at {first_ensure_s:.2f} s in {perf_counter() - file_started:.1f} s", flush=True)
        except Exception as error:
            qc_rows.append({"mouse": mouse, "date": session_date, "recording": label, "status": f"error: {error}"})
            print(f"        ERROR after {perf_counter() - file_started:.1f} s: {error}", flush=True)
if not tau_rows:
    raise ValueError("No recordings contained a usable Ensure event")
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    CACHE_FILE,
    relative_time_s=relative_time,
    tau_ns=np.vstack(tau_rows),
    delta_tau_ns=np.vstack(delta_tau_rows),
    lick_counts_per_bin=np.vstack(lick_count_rows),
    ensure_counts_per_bin=np.vstack(ensure_count_rows),
    bin_width_s=np.asarray(SAMPLE_INTERVAL_S),
    mouse=np.asarray([row["mouse"] for row in metadata_rows], dtype=str),
    session_date=np.asarray([row["date"] for row in metadata_rows], dtype=str),
    recording=np.asarray([row["recording"] for row in metadata_rows], dtype=str),
    source_file=np.asarray([row["source_file"] for row in metadata_rows], dtype=str),
    first_ensure_s=np.asarray([row["first_ensure_s"] for row in metadata_rows], dtype=float),
    baseline_tau_ns=np.asarray([row["baseline_tau_ns"] for row in metadata_rows], dtype=float),
    baseline_coverage=np.asarray([row["baseline_coverage"] for row in metadata_rows], dtype=float),
)
write_dict_rows(OUTPUT_FOLDER / "recording_qc.csv", qc_rows)
write_dict_rows(OUTPUT_FOLDER / "first_ensure_metadata.csv", metadata_rows)
print(f"Finished in {perf_counter() - processing_started:.1f} s", flush=True)
print(f"Cached {len(tau_rows)} first-Ensure transitions at {CACHE_FILE.resolve()}", flush=True)
qc_rows

## Fast cache reload and filtering

After creating the cache, future sessions only need the imports, configuration, and the cells below. Empty filters include all cached recordings.

In [ ]:
MOUSE_FILTER = []  # Example: ["SC43", "SC44"]
DATE_FILTER = []  # Example: ["250625"]
RECORDING_TEXT = ""  # Example: "FoodReward"
SHOW_LICKING = True
SHOW_ENSURE_SCHEDULE = True
HEATMAP_DISPLAY_BIN_S = 0.5  # Display only; cached data and mean traces remain at 0.1 s

if not CACHE_FILE.is_file():
    raise FileNotFoundError(f"Run the slow processing cell once to create {CACHE_FILE}")
with np.load(CACHE_FILE, allow_pickle=False) as cache:
    relative_time = cache["relative_time_s"]
    all_tau = cache["tau_ns"]
    all_delta_tau = cache["delta_tau_ns"]
    all_lick_counts = cache["lick_counts_per_bin"].astype(float)
    all_ensure_counts = cache["ensure_counts_per_bin"].astype(float)
    bin_width_s = float(cache["bin_width_s"])
    all_mice = cache["mouse"].astype(str)
    all_dates = cache["session_date"].astype(str)
    all_recordings = cache["recording"].astype(str)
    all_first_ensure = cache["first_ensure_s"].astype(float)
    all_baseline_coverage = cache["baseline_coverage"].astype(float)
selected = np.ones(all_delta_tau.shape[0], dtype=bool)
if MOUSE_FILTER:
    selected &= np.isin(all_mice, np.asarray(MOUSE_FILTER, dtype=str))
if DATE_FILTER:
    selected &= np.isin(all_dates, np.asarray(DATE_FILTER, dtype=str))
if RECORDING_TEXT:
    selected &= np.char.find(np.char.lower(all_recordings), RECORDING_TEXT.casefold()) >= 0
if not np.any(selected):
    raise ValueError("No cached recordings match the current filters")
tau = all_tau[selected]
delta_tau = all_delta_tau[selected]
lick_counts = all_lick_counts[selected]
ensure_counts = all_ensure_counts[selected]
recording_mice = all_mice[selected]
recording_dates = all_dates[selected]
recording_labels = all_recordings[selected]
first_ensure_times = all_first_ensure[selected]
baseline_coverage = all_baseline_coverage[selected]
print(f"Selected {delta_tau.shape[0]} of {all_delta_tau.shape[0]} cached recordings")
print(f"Mice: {', '.join(np.unique(recording_mice))}")
print(f"Recordings: {', '.join(np.unique(recording_labels))}")

In [ ]:
mouse_names = np.unique(recording_mice)
heatmap_block_size = max(1, int(round(HEATMAP_DISPLAY_BIN_S / bin_width_s)))
heatmap_slices = [slice(start, min(start + heatmap_block_size, relative_time.size)) for start in range(0, relative_time.size, heatmap_block_size)]
heatmap_time = np.asarray([np.mean(relative_time[section]) for section in heatmap_slices])
def block_mean(values):
    reduced = []
    for section in heatmap_slices:
        block = values[:, section]
        finite_count = np.sum(np.isfinite(block), axis=1)
        block_sum = np.nansum(block, axis=1)
        reduced.append(np.divide(block_sum, finite_count, out=np.full(block_sum.shape, np.nan), where=finite_count > 0))
    return np.column_stack(reduced)
def block_any(values):
    return np.column_stack([np.any(values[:, section] > 0, axis=1) for section in heatmap_slices]).astype(float)
lifetime_heatmap = block_mean(delta_tau)
lick_heatmap = block_any(lick_counts)
ensure_heatmap = block_any(ensure_counts)
mouse_lifetime_means = np.vstack([np.nanmean(delta_tau[recording_mice == mouse], axis=0) for mouse in mouse_names])
grand_lifetime_mean = np.nanmean(mouse_lifetime_means, axis=0)
lifetime_sem = (
    np.zeros_like(grand_lifetime_mean)
    if mouse_names.size == 1
    else np.nanstd(mouse_lifetime_means, axis=0, ddof=1) / np.sqrt(mouse_names.size)
)
panel_count = 2 + (2 if SHOW_LICKING else 0) + int(SHOW_ENSURE_SCHEDULE)
height_ratios = [1.2, 1.0] + ([1.0, 1.0] if SHOW_LICKING else []) + ([0.7] if SHOW_ENSURE_SCHEDULE else [])
figure, axes = plt.subplots(
    panel_count, 1, figsize=(12, 2.6 * panel_count), sharex=True, constrained_layout=True,
    gridspec_kw={"height_ratios": height_ratios},
)
axes = np.atleast_1d(axes)
figure.suptitle("Transition around first Ensure delivery")
image = axes[0].imshow(
    lifetime_heatmap, aspect="auto", origin="lower",
    extent=[heatmap_time[0] - HEATMAP_DISPLAY_BIN_S / 2, heatmap_time[-1] + HEATMAP_DISPLAY_BIN_S / 2, 0, delta_tau.shape[0]],
    cmap="coolwarm", interpolation="none", resample=False,
)
axes[0].set_ylabel("Recording")
figure.colorbar(image, ax=axes[0], label="Delta tau (ns)")
for trace in mouse_lifetime_means:
    axes[1].plot(relative_time, trace, color="0.75", linewidth=1)
axes[1].plot(relative_time, grand_lifetime_mean, color="tab:blue", linewidth=2.5, label=f"Mouse mean (n={mouse_names.size})")
axes[1].fill_between(relative_time, grand_lifetime_mean - lifetime_sem, grand_lifetime_mean + lifetime_sem, color="tab:blue", alpha=0.2)
axes[1].set_ylabel("Delta tau (ns)")
axes[1].legend()
next_axis = 2
if SHOW_LICKING:
    axes[next_axis].imshow(
        lick_heatmap, aspect="auto", origin="lower",
        extent=[heatmap_time[0] - HEATMAP_DISPLAY_BIN_S / 2, heatmap_time[-1] + HEATMAP_DISPLAY_BIN_S / 2, 0, lick_counts.shape[0]],
        cmap="Greys", vmin=0, vmax=1, interpolation="none", resample=False,
    )
    axes[next_axis].set_ylabel("Lick raster")
    next_axis += 1
    lick_rates = lick_counts / bin_width_s
    mouse_lick_means = np.vstack([np.nanmean(lick_rates[recording_mice == mouse], axis=0) for mouse in mouse_names])
    grand_lick_mean = np.nanmean(mouse_lick_means, axis=0)
    lick_sem = np.zeros_like(grand_lick_mean) if mouse_names.size == 1 else np.nanstd(mouse_lick_means, axis=0, ddof=1) / np.sqrt(mouse_names.size)
    for trace in mouse_lick_means:
        axes[next_axis].plot(relative_time, trace, color="0.75", linewidth=1)
    axes[next_axis].plot(relative_time, grand_lick_mean, color="tab:green", linewidth=2.5, label=f"Mouse mean (n={mouse_names.size})")
    axes[next_axis].fill_between(relative_time, grand_lick_mean - lick_sem, grand_lick_mean + lick_sem, color="tab:green", alpha=0.2)
    axes[next_axis].set_ylabel("Lick rate (Hz)")
    axes[next_axis].legend()
    next_axis += 1
if SHOW_ENSURE_SCHEDULE:
    axes[next_axis].imshow(
        ensure_heatmap, aspect="auto", origin="lower",
        extent=[heatmap_time[0] - HEATMAP_DISPLAY_BIN_S / 2, heatmap_time[-1] + HEATMAP_DISPLAY_BIN_S / 2, 0, ensure_counts.shape[0]],
        cmap="Oranges", vmin=0, vmax=1, interpolation="none", resample=False,
    )
    axes[next_axis].set_ylabel("Ensure raster")
for axis in axes:
    axis.axvline(0, color="tab:orange", linestyle="--")
axes[-1].set_xlabel("Time from first Ensure delivery (s)")
plt.show()

In [ ]:
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    OUTPUT_FOLDER / "first_ensure_transition_selected.npz",
    relative_time_s=relative_time,
    tau_ns=tau,
    delta_tau_ns=delta_tau,
    lick_counts_per_bin=lick_counts,
    ensure_counts_per_bin=ensure_counts,
    bin_width_s=np.asarray(bin_width_s),
    mouse=recording_mice,
    session_date=recording_dates,
    recording=recording_labels,
    first_ensure_s=first_ensure_times,
    baseline_coverage=baseline_coverage,
)
figure.savefig(OUTPUT_FOLDER / "first_ensure_transition.png", dpi=200)
print(f"Wrote selected results to {OUTPUT_FOLDER.resolve()}")

The Ensure raster makes subsequent scheduled deliveries explicit. Therefore, responses after time zero should be interpreted as the transition into the repeated-reward period, not as an isolated response to only one delivery.